In [1]:
# =============================================================================
# Insider Threat Behavioral Intelligence System
# Notebook : 06_model_export.ipynb
# =============================================================================

# Module 06: Model Packaging, Serialization & Production Verification

This notebook performs production export verification, artifact validation, and end-to-end inference testing:
- Model Artifact Inventory Audit
- JSON Packaging Manifest Generation (`models/model_manifest.json`)
- Production Scoring Pipeline Verification & Sanity Test

In [2]:
import json
import warnings
from pathlib import Path
import datetime

import joblib
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
print("Export verification utilities loaded!")

Export verification utilities loaded!


In [3]:
PROJECT_ROOT = Path("..").resolve()
MODEL_DIR = PROJECT_ROOT / "models"

expected_artifacts = [
    "scaler.pkl",
    "isolation_forest.pkl",
    "one_class_svm.pkl",
    "lof.pkl",
    "elliptic_envelope.pkl",
    "pca.pkl",
    "dbscan.pkl",
    "kmeans.pkl"
]

inventory = []
for artifact in expected_artifacts:
    path = MODEL_DIR / artifact
    exists = path.exists()
    size_kb = round(path.stat().st_size / 1024, 2) if exists else 0
    inventory.append({
        'Artifact': artifact,
        'Status': 'EXISTS' if exists else 'MISSING',
        'Size_KB': size_kb,
        'Path': str(path)
    })

inventory_df = pd.DataFrame(inventory)
print("Model Artifact Inventory:")
inventory_df

## 1. Export Packaging Manifest

In [4]:
manifest = {
    "project": "Insider Threat Behavioral Intelligence System",
    "export_timestamp": datetime.datetime.now().isoformat(),
    "version": "1.0.0",
    "models": [item['Artifact'] for item in inventory if item['Status'] == 'EXISTS'],
    "supported_features": [
        "logon_count",
        "after_hours_logon_count",
        "usb_connect_count",
        "file_copy_count",
        "email_external_count",
        "email_bcc_count",
        "http_job_search_count",
        "psychometric_N",
        "psychometric_O"
    ],
    "risk_thresholds": {
        "CRITICAL": 75.0,
        "HIGH": 50.0,
        "MEDIUM": 25.0,
        "LOW": 0.0
    }
}

manifest_path = MODEL_DIR / "model_manifest.json"
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=4)

print(f"Manifest created successfully at: {manifest_path}")

## 2. End-to-End Production Pipeline Sanity Check

In [5]:
# Load scaler and model artifacts to test real-time inference payload
scaler_path = MODEL_DIR / "scaler.pkl"
iso_path = MODEL_DIR / "isolation_forest.pkl"

if scaler_path.exists() and iso_path.exists():
    scaler = joblib.load(scaler_path)
    iso_model = joblib.load(iso_path)
    
    # Simulate single employee feature vector (9 features)
    dummy_sample = np.array([[45, 12, 8, 25, 60, 5, 2, 28.5, 32.1]])
    
    # Run scaling and inference
    scaled_sample = scaler.transform(dummy_sample)
    prediction = iso_model.predict(scaled_sample)
    score = -iso_model.score_samples(scaled_sample)[0] if hasattr(iso_model, "score_samples") else 0.5
    
    print("=== INFERENCE SANITY TEST SUCCESSFUL ===")
    print(f"Raw Features       : {dummy_sample.tolist()[0]}")
    print(f"Model Prediction   : {'ANOMALY (-1)' if prediction[0] == -1 else 'NORMAL (1)'}")
    print(f"Anomaly Raw Score  : {round(float(score), 4)}")
    print("========================================")
else:
    print("Inference test skipped: model files missing.")